# **1. Import Library**
Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [78]:
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# **2. Memuat Dataset dari Hasil Clustering**
Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [79]:
# Gunakan dataset hasil clustering yang memiliki fitur Target
# Silakan gunakan dataset data_clustering jika tidak menerapkan Interpretasi Hasil Clustering [Advanced]
# Silakan gunakan dataset data_clustering_inverse jika menerapkan Interpretasi Hasil Clustering [Advanced]

### MULAI CODE ###

df = pd.read_csv("data_clustering_inverse.csv")

### SELESAI CODE ###

In [80]:
# Tampilkan 5 baris pertama dengan function head

### MULAI CODE ###

df.head()

### SELESAI CODE ###

,TransactionAmount,TransactionType,Location,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,AgeGroupBin,Target
0,14.09,Debit,San Diego,ATM,70.0,Doctor,81.0,1.0,5112.21,tinggi,1
1,376.24,Debit,Houston,ATM,68.0,Doctor,141.0,1.0,13758.91,tinggi,0
2,126.29,Debit,Mesa,Online,19.0,Student,56.0,1.0,1122.35,rendah,1
3,184.50,Debit,Raleigh,Online,26.0,Student,25.0,1.0,8569.06,rendah,1
4,92.15,Debit,Oklahoma City,ATM,18.0,Student,172.0,1.0,781.68,rendah,1


## **(OPSIONAL) Feature Engineering: One Hot Encoding**
Langkah ini **HANYA** dilakukan saat Anda menggunakan **data hasil Inverse** untuk memenuhi kriteria Advanced pada latihan sebelumnya.

In [81]:
FEATURE_COLUMNS = [
    "TransactionAmount",
    "TransactionType",
    "Location",
    "Channel",
    "CustomerAge",
    "CustomerOccupation",
    "TransactionDuration",
    "LoginAttempts",
    "AccountBalance",
    "AgeGroupBin",
]

CATEGORICAL_COLUMNS = [
    "TransactionType",
    "Location",
    "Channel",
    "CustomerOccupation",
    "AgeGroupBin",
]

NUMERICAL_COLUMNS = [
    "TransactionAmount",
    "CustomerAge",
    "TransactionDuration",
    "LoginAttempts",
    "AccountBalance",
]

In [82]:
df = df.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)

# **3. Data Splitting**
Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [83]:
X = df[FEATURE_COLUMNS].copy()
y = df["Target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1945, 10)
y shape: (1945,)


In [84]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (1556, 10)
Testing data: (389, 10)


In [85]:
one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

## Preprocessing

In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            one_hot_encoder,
            CATEGORICAL_COLUMNS,
        ),
        (
            "numerical",
            "passthrough",
            NUMERICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# **4. Membangun Model Klasifikasi**
Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Menggunakan algoritma klasifikasi yaitu Decision Tree.
2. Latih model menggunakan data yang sudah dipisah.

In [87]:
classification_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
            ),
        ),
    ]
)

classification_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['TransactionType',
                                                   'Location', 'Channel',
                                                   'CustomerOccupation',
                                                   'AgeGroupBin']),
                                                 ('numerical', 'passthrough',
                                                  ['TransactionAmount',
                                                   'CustomerAge',
                                                   'TransactionDuration',
                                                   'LoginAttempts',
                                                   'AccountBalance'])],
                                   verbose_feature_names_out=False)),
                ('model', RandomForestClassifier(random_state=42))])

In [88]:
classification_pipeline.fit(
    X_train,
    y_train,
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['TransactionType',
                                                   'Location', 'Channel',
                                                   'CustomerOccupation',
                                                   'AgeGroupBin']),
                                                 ('numerical', 'passthrough',
                                                  ['TransactionAmount',
                                                   'CustomerAge',
                                                   'TransactionDuration',
                                                   'LoginAttempts',
                                                   'AccountBalance'])],
                                   verbose_feature_names_out=False)),
                ('model', RandomForestClassifier(random_state=42))])

### Evaluation

In [89]:
y_prediction = classification_pipeline.predict(
    X_test
)

accuracy = accuracy_score(
    y_test,
    y_prediction,
)

print("Accuracy:", accuracy)

print(
    classification_report(
        y_test,
        y_prediction,
    )
)

print(
    "Confusion matrix:\n",
    confusion_matrix(
        y_test,
        y_prediction,
    ),
)

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       196
           1       1.00      1.00      1.00       193

    accuracy                           1.00       389
   macro avg       1.00      1.00      1.00       389
weighted avg       1.00      1.00      1.00       389

Confusion matrix:
 [[196   0]
 [  0 193]]


In [90]:
fitted_preprocessor = (
    classification_pipeline.named_steps[
        "preprocessor"
    ]
)

In [91]:
encoded_values = fitted_preprocessor.transform(
    X_test.head()
)

encoded_columns = (
    fitted_preprocessor.get_feature_names_out()
)

encoded_df = pd.DataFrame(
    encoded_values,
    columns=encoded_columns,
    index=X_test.head().index,
)

encoded_df.head()

,TransactionType_Credit,TransactionType_Debit,Location_Albuquerque,Location_Atlanta,Location_Austin,Location_Baltimore,Location_Boston,Location_Charlotte,Location_Chicago,Location_Colorado Springs,...,CustomerOccupation_Retired,CustomerOccupation_Student,AgeGroupBin_rendah,AgeGroupBin_sedang,AgeGroupBin_tinggi,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance
687,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,183.32,21.0,74.0,1.0,137.42
1690,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,162.10,41.0,151.0,1.0,8576.65
1187,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,77.50,51.0,201.0,1.0,11079.77
305,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,123.85,56.0,244.0,1.0,3959.62
1932,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,603.54,44.0,26.0,1.0,10517.47


In [92]:
encoded_values = fitted_preprocessor.transform(
    X_test.head()
)

encoded_columns = (
    fitted_preprocessor.get_feature_names_out()
)

encoded_df = pd.DataFrame(
    encoded_values,
    columns=encoded_columns,
    index=X_test.head().index,
)

encoded_df.head()

,TransactionType_Credit,TransactionType_Debit,Location_Albuquerque,Location_Atlanta,Location_Austin,Location_Baltimore,Location_Boston,Location_Charlotte,Location_Chicago,Location_Colorado Springs,...,CustomerOccupation_Retired,CustomerOccupation_Student,AgeGroupBin_rendah,AgeGroupBin_sedang,AgeGroupBin_tinggi,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance
687,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.0,0.0,183.32,21.0,74.0,1.0,137.42
1690,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,162.10,41.0,151.0,1.0,8576.65
1187,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,77.50,51.0,201.0,1.0,11079.77
305,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,123.85,56.0,244.0,1.0,3959.62
1932,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,603.54,44.0,26.0,1.0,10517.47


In [93]:
print(
    "Jumlah fitur setelah encoding:",
    encoded_df.shape[1],
)

print(encoded_columns)

Jumlah fitur setelah encoding: 60
['TransactionType_Credit' 'TransactionType_Debit' 'Location_Albuquerque'
 'Location_Atlanta' 'Location_Austin' 'Location_Baltimore'
 'Location_Boston' 'Location_Charlotte' 'Location_Chicago'
 'Location_Colorado Springs' 'Location_Columbus' 'Location_Dallas'
 'Location_Denver' 'Location_Detroit' 'Location_El Paso'
 'Location_Fort Worth' 'Location_Fresno' 'Location_Houston'
 'Location_Indianapolis' 'Location_Jacksonville' 'Location_Kansas City'
 'Location_Las Vegas' 'Location_Los Angeles' 'Location_Louisville'
 'Location_Memphis' 'Location_Mesa' 'Location_Miami' 'Location_Milwaukee'
 'Location_Nashville' 'Location_New York' 'Location_Oklahoma City'
 'Location_Omaha' 'Location_Philadelphia' 'Location_Phoenix'
 'Location_Portland' 'Location_Raleigh' 'Location_Sacramento'
 'Location_San Antonio' 'Location_San Diego' 'Location_San Francisco'
 'Location_San Jose' 'Location_Seattle' 'Location_Tucson'
 'Location_Virginia Beach' 'Location_Washington' 'Channel_AT

In [94]:
joblib.dump(
    classification_pipeline,
    "classification_pipeline.joblib",
)

['classification_pipeline.joblib']

In [95]:
loaded_pipeline = joblib.load(
    "classification_pipeline.joblib"
)

loaded_prediction = loaded_pipeline.predict(
    X_test.head()
)

print(loaded_prediction)

[1 1 0 0 0]


In [96]:
original_prediction = (
    classification_pipeline.predict(
        X_test.head()
    )
)

print(
    original_prediction.tolist()
    == loaded_prediction.tolist()
)

True


In [97]:
new_customer = pd.DataFrame(
    [
        {
            "TransactionAmount": 100.0,
            "TransactionType": "Debit",
            "Location": "San Diego",
            "Channel": "ATM",
            "CustomerAge": 30,
            "CustomerOccupation": "Doctor",
            "TransactionDuration": 100.0,
            "LoginAttempts": 1,
            "AccountBalance": 5000.0,
            "AgeGroupBin": "rendah",
        }
    ]
)

prediction = loaded_pipeline.predict(
    new_customer
)

print("Predicted class:", prediction[0])

Predicted class: 1


In [98]:
joblib.dump(
    classification_pipeline,
    "classification_pipeline.joblib",
)

['classification_pipeline.joblib']

End of Code